# Citi Field TLC Pickup EDA — Ingestion and Event Analysis

This notebook uses the project ingestion and processing modules to load local TLC parquet files, filter the Citi Field pickup catchment (zones 93 and 253), create 15-minute counts, and compare pickups around Mets games.

## 1. Imports and file paths

Because this notebook is stored in `notebooks/`, the parquet folder is one level above it.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CITI_FIELD_EVENTS_PATH,
    CITI_FIELD_PICKUP_ZONES,
    get_citi_field_tlc_dir,
)
from src.ingestion.tlc_ingest import find_trip_files, load_pickup_columns
from src.processing.clean_tlc import aggregate_pickups_15min, clean_pickups
from src.processing.merge_events import (
    label_pickup_intervals,
    prepare_game_event_windows,
)

DATA_DIR = get_citi_field_tlc_dir()
CITI_FIELD_ZONES = CITI_FIELD_PICKUP_ZONES
files = find_trip_files(DATA_DIR)

print("TLC parquet folder:", DATA_DIR)
print("Number of files:", len(files))
for file in files:
    print(file.name)

## 2. Inspect one file

This confirms the available fields and data types before processing every month.

In [12]:
sample_file = files[0]
table = pq.ParquetFile(sample_file)

print("Columns:")
print(table.schema.names)

sample = pd.read_parquet(
    sample_file,
    columns=["pickup_datetime", "PULocationID", "DOLocationID"]
)

print(sample.head())
print(sample.dtypes)

Columns:
['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag']
      pickup_datetime  PULocationID  DOLocationID
0 2024-01-01 00:28:08           161           158
1 2024-01-01 00:12:53           137            79
2 2024-01-01 00:23:05            79           186
3 2024-01-01 00:41:04           234           148
4 2024-01-01 00:57:21           148            97
pickup_datetime    datetime64[us]
PULocationID                int32
DOLocationID                int32
dtype: object


## 3. Check monthly coverage and timestamp quality

In [13]:
monthly_checks = []

for file in files:
    month_df = pd.read_parquet(
        file,
        columns=["pickup_datetime", "PULocationID"]
    )
    month_df["pickup_datetime"] = pd.to_datetime(
        month_df["pickup_datetime"], errors="coerce"
    )
    monthly_checks.append({
        "file": file.name,
        "rows": len(month_df),
        "start": month_df["pickup_datetime"].min(),
        "end": month_df["pickup_datetime"].max(),
        "missing_timestamps": month_df["pickup_datetime"].isna().sum(),
    })

monthly_checks = pd.DataFrame(monthly_checks)
display(monthly_checks)

,file,rows,start,end,missing_timestamps
0,fhvhv_tripdata_2024-01.parquet,19663930,2024-01-01,2024-01-31 23:59:59,0
1,fhvhv_tripdata_2024-02.parquet,19359148,2024-02-01,2024-02-29 23:59:59,0
2,fhvhv_tripdata_2024-03.parquet,21280788,2024-03-01,2024-03-31 23:59:59,0
3,fhvhv_tripdata_2024-04.parquet,19733038,2024-04-01,2024-04-30 23:59:59,0
4,fhvhv_tripdata_2024-05.parquet,20704538,2024-05-01,2024-05-31 23:59:59,0
5,fhvhv_tripdata_2024-06.parquet,20123226,2024-06-01,2024-06-30 23:59:59,0
6,fhvhv_tripdata_2024-07.parquet,19182934,2024-07-01,2024-07-31 23:59:59,0
7,fhvhv_tripdata_2024-08.parquet,19128392,2024-08-01,2024-08-31 23:59:59,0
8,fhvhv_tripdata_2024-09.parquet,19209788,2024-09-01,2024-09-30 23:59:59,0
9,fhvhv_tripdata_2024-10.parquet,20028282,2024-10-01,2024-10-31 23:59:59,0


## 4. Count pickups in the Citi Field catchment by month

This checks both pickup zones used for the Citi Field catchment: zone 93 at Citi Field and nearby zone 253.

In [ ]:
zone_checks = []

for file in files:
    zone_df = pd.read_parquet(file, columns=["PULocationID"])
    zone_counts = zone_df["PULocationID"].value_counts()
    zone_checks.append({
        "file": file.name,
        "zone_93_pickups": int(zone_counts.get(93, 0)),
        "zone_253_pickups": int(zone_counts.get(253, 0)),
        "combined_catchment_pickups": int(
            zone_counts.reindex(CITI_FIELD_ZONES, fill_value=0).sum()
        ),
    })

zone_checks = pd.DataFrame(zone_checks)
display(zone_checks)

## Preparing Citi Field Catchment Pickup Data

This section uses the ingestion and cleaning modules to load monthly TLC pickups, filter to zones 93 and 253, and combine the records. The pickup zone is retained so the two zones can also be examined separately.

In [ ]:
raw_citi_trips = load_pickup_columns(files, CITI_FIELD_ZONES)
citi_trips = clean_pickups(raw_citi_trips, CITI_FIELD_ZONES)

print("Citi Field catchment pickups:", len(citi_trips))
print("Pickup counts by TLC zone:")
display(citi_trips["PULocationID"].value_counts().sort_index())
display(citi_trips.head())

## 6. Create 15-minute pickup counts

The processing module bins each pickup into a 15-minute interval. The combined count includes zones 93 and 253; a second table retains separate counts for each zone.

In [ ]:
pickup_counts, pickup_counts_by_zone = aggregate_pickups_15min(citi_trips)

print("Combined catchment counts:")
display(pickup_counts.head())
print("Counts by zone:")
display(pickup_counts_by_zone.head())
print("Highest-volume combined 15-minute intervals:")
display(pickup_counts.sort_values("pickup_count", ascending=False).head(10))

## 7. Check for Missing Values


In [ ]:
events_file = CITI_FIELD_EVENTS_PATH
events = pd.read_csv(events_file)

missing_by_column = events.isna().sum()
print("Missing values by column:")
display(missing_by_column)
print("Total missing entries:", int(missing_by_column.sum()))

## 8. Prepare Event Time Ranges

The merge processing module prepares Mets game windows from 30 minutes before first pitch through 90 minutes after each game ends. Concert rows are excluded from this game-window analysis until their durations are available.

In [ ]:
game_events = prepare_game_event_windows(events)
duplicate_ids = game_events["event_id"].duplicated().sum()
wrong_venue = (game_events["venue"] != "Citi Field").sum()

print("Invalid or reversed time ranges: 0 (validated by the processor)")
print("Duplicate game event IDs:", int(duplicate_ids))
print("Games outside Citi Field:", int(wrong_venue))

display(
    game_events[
        ["event_id", "event_date", "opponent", "start_time_dt",
         "end_time_dt", "window_start", "window_end"]
    ].head()
)

## 9. Match Pickup Counts to Events

The merge processing module compares each combined 15-minute catchment pickup count with the prepared Mets game times. Intervals are labeled `before_game`, `during_game`, `after_game`, or `no_event`. Matched intervals retain the event ID, game date, and opponent.

In [ ]:
pickup_event_matches = label_pickup_intervals(pickup_counts, game_events)

print("Pickup intervals by event phase:")
display(pickup_event_matches["event_phase"].value_counts())
display(
    pickup_event_matches[
        ["time_bin", "pickup_count", "event_id", "event_date",
         "opponent", "event_phase"]
    ].query("event_phase != 'no_event'").head(10)
)

## 10. Compare Pickup Counts by Game Phase

This section summarizes and compares the observed 15-minute catchment pickup counts before, during, and after Mets games. It reports the average and median for each phase and plots the average counts. Because `pickup_counts` includes only intervals with at least one recorded pickup, these summaries do not include zero-pickup intervals.

In [ ]:
import matplotlib.pyplot as plt

phase_order = ["before_game", "during_game", "after_game"]
game_phase_counts = pickup_event_matches[
    pickup_event_matches["event_phase"].isin(phase_order)
]

phase_summary = (
    game_phase_counts
    .groupby("event_phase")["pickup_count"]
    .agg(intervals="count", average_pickups="mean", median_pickups="median")
    .reindex(phase_order)
)

print("Pickup counts by event phase:")
display(phase_summary.round(2))

ax = phase_summary["average_pickups"].plot(
    kind="bar",
    figsize=(8, 5),
    color=["#6baed6", "#2171b5", "#6baed6"],
    rot=0,
)
ax.set_title("Average 15-Minute Citi Field Catchment Pickups by Mets Game Phase")
ax.set_xlabel("Game phase")
ax.set_ylabel("Average catchment pickups per observed 15-minute interval")
plt.tight_layout()
plt.show()